In [ ]:
# Colab/bootstrap: clone this repository and install it editable.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
MARK_REL = Path("src") / "mindscopex_analysis" / "__init__.py"


def find_repo_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / MARK_REL).is_file():
            return path
    return None


root = find_repo_root()
if root is None:
    workdir = Path(os.environ.get("COLAB_REPO_DIR", "/content/mindscopex_analysis"))
    if (workdir / MARK_REL).is_file():
        subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=False)
        root = workdir
    else:
        workdir.parent.mkdir(parents=True, exist_ok=True)
        if workdir.exists():
            shutil.rmtree(workdir)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(workdir)])
        root = workdir

os.environ["MINDSCOPEX_ROOT"] = str(root.resolve())
os.chdir(root)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print("ready:", root)


# 00. Qwen CRT 실제 텍스트 답변

Qwen 모델군에 CRT 문제를 직접 제시하고, thinking/non-thinking 모드에서 생성되는 전체 텍스트를 확인하는 기준 실험입니다. 이 결과는 이후 activation 및 lure feature 실험에서 어떤 모델과 조건을 우선 분석할지 정하는 행동 수준의 baseline입니다.

## 실행 전 고려사항

1. 기본 모델은 instruction-tuned `Qwen3-0.6B`, `1.7B`, `4B`이며 한 번에 하나만 GPU에 올립니다.
2. 샘플링은 [Qwen3 모델 카드](https://huggingface.co/Qwen/Qwen3-1.7B)의 권장값을 사용하고 seed를 기록합니다. 샘플링 결과는 seed에 따라 달라질 수 있습니다.
3. `answer_label`은 최종 답변 텍스트의 단순 문자열 판정입니다. `both`와 `other`는 반드시 원문을 직접 확인합니다.
4. Qwen-Scope 대상인 `Qwen3-1.7B-Base`는 instruction-tuned 채팅 모델이 아닙니다. 선택 옵션으로만 실행하며 결과를 같은 종류의 대화 성능으로 해석하지 않습니다.
5. `truncated=True`이면 reasoning이 끝나지 않은 것이므로 `MAX_NEW_TOKENS`를 늘려 다시 실행합니다.


In [ ]:
import os
import sys
from pathlib import Path

root = Path(os.environ.get("MINDSCOPEX_ROOT", Path.cwd())).resolve()
if not (root / "src" / "mindscopex_analysis" / "__init__.py").is_file():
    for candidate in [root, *root.parents]:
        if (candidate / "src" / "mindscopex_analysis" / "__init__.py").is_file():
            root = candidate
            break
    else:
        raise RuntimeError("Could not find repository root. Run the clone cell first.")

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(root)


In [ ]:
from collections import Counter
from html import escape

from IPython.display import HTML, display

from mindscopex_analysis import (
    DEFAULT_QWEN_CHAT_MODEL_IDS,
    clear_device_cache,
    crt_transfer_cases,
    generate_crt_response_suite,
    load_qwen_text_generation_model,
    recommended_dtype_name,
    save_qwen_text_responses,
)


def display_records(rows, columns):
    head = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
    body = []
    for row in rows:
        cells = "".join(f"<td>{escape(str(row.get(column, '')))}</td>" for column in columns)
        body.append(f"<tr>{cells}</tr>")
    table = (
        "<div style='overflow-x:auto'><table style='border-collapse:collapse'>"
        f"<thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"
    )
    display(HTML(table))


In [ ]:
MODEL_SPECS = [
    {
        "model_id": model_id,
        "thinking_modes": (False, True),
        "use_chat_template": True,
    }
    for model_id in DEFAULT_QWEN_CHAT_MODEL_IDS
]

INCLUDE_QWEN_SCOPE_BASE = False
if INCLUDE_QWEN_SCOPE_BASE:
    MODEL_SPECS.append(
        {
            "model_id": "Qwen/Qwen3-1.7B-Base",
            "thinking_modes": (None,),
            "use_chat_template": False,
        }
    )

CASES = crt_transfer_cases()
DTYPE = recommended_dtype_name()
MAX_NEW_TOKENS = 1024
DO_SAMPLE = True
SEED = 42
SYSTEM_PROMPT = ""
OUTPUT_PATH = root / "outputs" / "00_qwen_crt_text_responses.json"

print({
    "models": [spec["model_id"] for spec in MODEL_SPECS],
    "cases": [case.case_id for case in CASES],
    "dtype": DTYPE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": DO_SAMPLE,
    "seed": SEED,
})


In [ ]:
case_rows = [
    {
        "case": case.case_id,
        "family": case.family,
        "correct": case.correct_answer.strip(),
        "lure": case.lure_answer.strip(),
        "prompt": case.prompt,
    }
    for case in CASES
]
display_records(case_rows, ["case", "family", "correct", "lure", "prompt"])


In [ ]:
responses = []

for spec in MODEL_SPECS:
    model_id = spec["model_id"]
    print(f"\nLoading {model_id} ...")
    model, tokenizer = load_qwen_text_generation_model(
        model_id,
        device_map="auto",
        dtype=DTYPE,
    )

    model_responses = generate_crt_response_suite(
        model,
        tokenizer,
        CASES,
        model_id=model_id,
        thinking_modes=spec["thinking_modes"],
        use_chat_template=spec["use_chat_template"],
        system_prompt=SYSTEM_PROMPT,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        seed=SEED,
    )
    responses.extend(model_responses)
    save_qwen_text_responses(responses, OUTPUT_PATH)

    for response in model_responses:
        preview = response.answer.replace("\n", " ")[:120]
        print(f"[{response.case_id} | {response.mode} | {response.answer_label}] {preview}")

    del model_responses, model, tokenizer
    clear_device_cache()

print(f"\nsaved {len(responses)} responses to {OUTPUT_PATH}")


In [ ]:
summary_rows = []
for response in responses:
    row = response.summary_row()
    row["final_answer"] = row["final_answer"].replace("\n", " ")[:200]
    summary_rows.append(row)

display_records(
    summary_rows,
    ["model", "case", "mode", "label", "final_answer", "output_tokens", "seconds", "truncated"],
)


In [ ]:
SELECT_MODEL = ""  # 예: Qwen3-1.7B, 빈 문자열이면 전체
SELECT_CASE = ""   # 예: bat_ball_original, 빈 문자열이면 전체
SHOW_THINKING = True

selected = [
    response
    for response in responses
    if (not SELECT_MODEL or response.model_id.endswith(SELECT_MODEL))
    and (not SELECT_CASE or response.case_id == SELECT_CASE)
]

for response in selected:
    heading = (
        f"{response.model_id} | {response.case_id} | "
        f"{response.mode} | {response.answer_label}"
    )
    sections = [f"<h3>{escape(heading)}</h3>"]
    if SHOW_THINKING and response.thinking:
        sections.append(f"<h4>Thinking</h4><pre>{escape(response.thinking)}</pre>")
    sections.append(f"<h4>Final answer</h4><pre>{escape(response.answer)}</pre>")
    display(HTML("".join(sections)))


In [ ]:
counts = Counter(
    (response.model_id.rsplit("/", 1)[-1], response.mode, response.answer_label)
    for response in responses
)
aggregate_rows = [
    {"model": model, "mode": mode, "label": label, "count": count}
    for (model, mode, label), count in sorted(counts.items())
]
display_records(aggregate_rows, ["model", "mode", "label", "count"])


## 결과를 읽는 순서

1. 먼저 `truncated`가 없는지 확인합니다. 잘린 응답은 정오 판정에서 제외합니다.
2. `both`는 정답과 함정 답을 설명 과정에서 함께 언급한 경우가 많으므로 최종 결론을 직접 읽습니다.
3. 같은 모델의 thinking/non-thinking 차이를 비교해 reasoning이 함정 답을 수정하는지 확인합니다.
4. 작은 모델에서만 반복되는 오답과 모델 크기에 무관하게 반복되는 오답을 구분합니다.
5. 이후 feature 실험은 함정 답이 실제로 관찰되거나, thinking 여부에 따라 답이 바뀌는 모델/문항 조합을 우선 대상으로 삼습니다.
6. 확률적 결과를 논문에 사용할 때는 `SEED`를 여러 개로 늘려 정답률과 함정 답률의 평균 및 신뢰구간을 계산합니다.
